In [45]:
import pandas as pd

# Load the dataset into a pandas DataFrame
df = pd.read_csv(r"C:\Users\Chetti Rishika\Downloads\iot_sensor.csv")

# Display the first 5 rows of the DataFrame
print("First 5 rows of the DataFrame:")
print(df.head())

# Print the shape of the DataFrame
print("\nShape of the DataFrame:")
print(df.shape)

# Display the column names and their data types
print("\nColumn names and their data types:")
print(df.info())

First 5 rows of the DataFrame:
             timestamp sensor_id  temperature  humidity
0  2025-02-01 00:00:00        S2         24.0      40.0
1  2025-02-01 01:00:00        S3         30.0       NaN
2  2025-02-01 02:00:00        S1         24.0      50.0
3  2025-02-01 03:00:00        S2         24.0       NaN
4  2025-02-01 04:00:00        S3         23.0      42.0

Shape of the DataFrame:
(50, 4)

Column names and their data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   timestamp    50 non-null     object 
 1   sensor_id    50 non-null     object 
 2   temperature  41 non-null     float64
 3   humidity     39 non-null     float64
dtypes: float64(2), object(2)
memory usage: 1.7+ KB
None


In [46]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by='timestamp').reset_index(drop=True)

print("Timestamp column converted to datetime and DataFrame sorted.")
print(df.head())

Timestamp column converted to datetime and DataFrame sorted.
            timestamp sensor_id  temperature  humidity
0 2025-02-01 00:00:00        S2         24.0      40.0
1 2025-02-01 01:00:00        S3         30.0       NaN
2 2025-02-01 02:00:00        S1         24.0      50.0
3 2025-02-01 03:00:00        S2         24.0       NaN
4 2025-02-01 04:00:00        S3         23.0      42.0


In [47]:
print("Missing values before ffill:")
print(df.isnull().sum())

# Identify columns with missing values and apply ffill
df['temperature'] = df['temperature'].ffill()
df['humidity'] = df['humidity'].ffill()

print("\nMissing values after ffill:")
print(df.isnull().sum())
print("\nDataFrame after ffill:")
print(df.head())

Missing values before ffill:
timestamp       0
sensor_id       0
temperature     9
humidity       11
dtype: int64

Missing values after ffill:
timestamp      0
sensor_id      0
temperature    0
humidity       0
dtype: int64

DataFrame after ffill:
            timestamp sensor_id  temperature  humidity
0 2025-02-01 00:00:00        S2         24.0      40.0
1 2025-02-01 01:00:00        S3         30.0      40.0
2 2025-02-01 02:00:00        S1         24.0      50.0
3 2025-02-01 03:00:00        S2         24.0      50.0
4 2025-02-01 04:00:00        S3         23.0      42.0


In [48]:
df['temperature_smoothed'] = df['temperature'].rolling(window=3).mean()
df['humidity_smoothed'] = df['humidity'].rolling(window=3).mean()

print("DataFrame with smoothed temperature and humidity:")
print(df.head())

DataFrame with smoothed temperature and humidity:
            timestamp sensor_id  temperature  humidity  temperature_smoothed  \
0 2025-02-01 00:00:00        S2         24.0      40.0                   NaN   
1 2025-02-01 01:00:00        S3         30.0      40.0                   NaN   
2 2025-02-01 02:00:00        S1         24.0      50.0             26.000000   
3 2025-02-01 03:00:00        S2         24.0      50.0             26.000000   
4 2025-02-01 04:00:00        S3         23.0      42.0             23.666667   

   humidity_smoothed  
0                NaN  
1                NaN  
2          43.333333  
3          46.666667  
4          47.333333  


In [49]:
from sklearn.preprocessing import StandardScaler

# Identify numerical sensor reading columns for normalization
numerical_cols = ['temperature_smoothed', 'humidity_smoothed']

print(f"Identified numerical columns for normalization: {numerical_cols}")

Identified numerical columns for normalization: ['temperature_smoothed', 'humidity_smoothed']


In [50]:
scaler = StandardScaler()
df[['temperature_normalized', 'humidity_normalized']] = scaler.fit_transform(df[numerical_cols])

print("DataFrame with normalized temperature and humidity:")
print(df.head())

DataFrame with normalized temperature and humidity:
            timestamp sensor_id  temperature  humidity  temperature_smoothed  \
0 2025-02-01 00:00:00        S2         24.0      40.0                   NaN   
1 2025-02-01 01:00:00        S3         30.0      40.0                   NaN   
2 2025-02-01 02:00:00        S1         24.0      50.0             26.000000   
3 2025-02-01 03:00:00        S2         24.0      50.0             26.000000   
4 2025-02-01 04:00:00        S3         23.0      42.0             23.666667   

   humidity_smoothed  temperature_normalized  humidity_normalized  
0                NaN                     NaN                  NaN  
1                NaN                     NaN                  NaN  
2          43.333333                1.440821            -0.388474  
3          46.666667                1.440821             0.915494  
4          47.333333               -0.217110             1.176287  


In [51]:
df.drop(columns=['temperature', 'humidity', 'temperature_smoothed', 'humidity_smoothed'], inplace=True)

print("DataFrame after dropping original and smoothed columns:")
print(df.head())

DataFrame after dropping original and smoothed columns:
            timestamp sensor_id  temperature_normalized  humidity_normalized
0 2025-02-01 00:00:00        S2                     NaN                  NaN
1 2025-02-01 01:00:00        S3                     NaN                  NaN
2 2025-02-01 02:00:00        S1                1.440821            -0.388474
3 2025-02-01 03:00:00        S2                1.440821             0.915494
4 2025-02-01 04:00:00        S3               -0.217110             1.176287


In [52]:
from sklearn.preprocessing import OneHotEncoder

print("OneHotEncoder imported.")

OneHotEncoder imported.


In [53]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_sensor_ids = encoder.fit_transform(df[['sensor_id']])
encoded_df = pd.DataFrame(encoded_sensor_ids, columns=encoder.get_feature_names_out(['sensor_id']))

df = pd.concat([df.drop(columns=['sensor_id']), encoded_df], axis=1)

print("DataFrame after one-hot encoding 'sensor_id' column:")
print(df.head())

DataFrame after one-hot encoding 'sensor_id' column:
            timestamp  temperature_normalized  humidity_normalized  \
0 2025-02-01 00:00:00                     NaN                  NaN   
1 2025-02-01 01:00:00                     NaN                  NaN   
2 2025-02-01 02:00:00                1.440821            -0.388474   
3 2025-02-01 03:00:00                1.440821             0.915494   
4 2025-02-01 04:00:00               -0.217110             1.176287   

   sensor_id_S1  sensor_id_S2  sensor_id_S3  
0           0.0           1.0           0.0  
1           0.0           0.0           1.0  
2           1.0           0.0           0.0  
3           0.0           1.0           0.0  
4           0.0           0.0           1.0  
